之前我们了解了checkpoints（检查点）的功能。Checkpoints不仅仅是让我们可以保存Graph执行过程中的每一步状态，更重要的是它实现了LangGraph的持久化层（persistence layer）。
当我们在编译Graph时引入checkpointer，它会在每个super-step自动保存Graph状态的快照。这些checkpoints被保存在Thread（线程）中，使得我们可以在Graph执行完成后仍然访问Graph的状态。
正是因为这个持久化层，很多强大的功能才成为可能，包括Human-in-the-loop的人机交互、记忆（Memory）、时间旅行（Time Travel）和容错（Fault-tolerance）等。
基于持久化层有一个强大的功能--Interrupts（中断），它允许我们在Graph执行的特定位置暂停，等待外部输入后再继续执行，这为实现Human-in-the-loop模式提供了基础。
### 1.Interrupts功能概述
有一些AI编程助手，当AI需要执行某些关键操作（比如修改当前文件、允许可能影响系统的命令）时，会暂停并询问你是否确认呢，只有在你批准后才能继续执行。
在日常生活中，类似的情景非常普遍：
- 在执行关键操作（比如发送邮件、转账、操作数据）之前，需要人工审批
- 在LLM生成内容后，需要人工审查和编辑
- 在执行工具调用前，需要人工确认和修改参数
- 需要收集用户输入，并在输入无效时重新询问
为了解决这些问题，LangGraph提供了Interrupts，它允许Graph在执行过程中暂停，等待外部输入（比如人工审批、用户确认呢等），然后根据输入决定如何继续执行。
Interrupts的核心价值不仅仅在于暂停Graph的执行，更重要的是它实现了对Graph State的高级控制：
- 暂停执行：在Graph执行的任何位置暂停，保存当前State的完整快照到持久层
- 接收外部输入：在暂停期间，可以接收来自外部的输入（人工审批、用户输入、系统决策等）
- 基于输入继续执行：外部输入会被传递回Graph，成为interrupt()调用的返回值，Graph可以基于这个输入值来决定后续的执行路径和State的更新方式
- 动态控制State：通过外部输入，我们可以修改State的值、改变执行路由、甚至重新定义Graph的行为，这是对State的一种高级控制能力